In [1]:
import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from pathlib import Path
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import random
from torch.optim import AdamW
import numpy as np
from jiwer import wer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = random.Random(12)

# SNR Augmentation

In [2]:
def mix_with_noise(clean, noise, snr_db):
    # Convert stereo to mono if necessary
    if clean.shape[0] > 1:
        clean = clean.mean(dim=0, keepdim=True)

    if noise.shape[0] > 1:
        noise = noise.mean(dim=0, keepdim=True)

    clean_length = clean.shape[1]
    noise_length = noise.shape[1]

    # Repeat noise if it is shorter than speech
    if noise_length < clean_length:
        repeats = (clean_length // noise_length) + 1
        noise = noise.repeat(1, repeats)

    # Crop noise to exactly match speech
    noise = noise[:, :clean_length]

    # Scale noise to desired SNR
    clean_rms = torch.sqrt(torch.mean(clean ** 2))
    noise_rms = torch.sqrt(torch.mean(noise ** 2))

    snr_linear = 10 ** (snr_db / 20)

    noise = noise * (
        clean_rms / (snr_linear * noise_rms + 1e-8)
    )

    return clean + noise

# SNR Files

One random noise file from each location ID

In [3]:
df = pd.read_csv(r"noise_meta_tr.csv")

random_noise = df.groupby("Location ID", group_keys=False).sample(
    n=1,
    random_state=42
)

noise_root = Path(r"tr")
noise_files = [noise_root / file for file in random_noise["utterance_id"]]

# Load  Data

In [4]:
def load_data(root):
    samples = []
    
    for trans_file in root.rglob("*.trans.txt"):
        with open(trans_file, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split(" ", 1)
                file_id = parts[0]
                transcript = parts[1]
    
                audio_path = trans_file.parent / f"{file_id}.flac"
    
                samples.append({
                    "audio": str(audio_path),
                    "text": transcript
                })
    return samples

root = Path(r"dev-clean")
test_root = Path(r"test-clean")

dev_samples = load_data(root)
test_samples = load_data(test_root)
print(len(test_samples))

2620


In [5]:
#TESTING
# dev_samples = rng.sample(dev_samples, k= 50)
# test_samples = rng.sample(test_samples, k= 25)

# Wave2Vec2 Testing

In [ ]:
#load
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

model = model.to(device)

audio_path = dev_samples[0]["audio"]
speech, sample_rate = torchaudio.load(audio_path)

if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    speech = resampler(speech)
    sample_rate = 16000

input_values = processor(speech.squeeze(), sampling_rate=sample_rate, return_tensors="pt").input_values.to(device)

with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids.cpu())

output = transcription[0]
print("Transcription:", transcription[0])

# Datasets

In [7]:
class SpeechDataset(Dataset):
    def __init__(self, samples, noise_files=None, snrs=None, augment=False):
        self.samples = samples
        self.noise_files = noise_files
        self.snrs = snrs
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        speech, sr = torchaudio.load(sample["audio"])

        if sr != 16000:
            speech = torchaudio.functional.resample(
                speech, sr, 16000
            )

        if self.augment:
            noise_path = rng.choice(self.noise_files)
            noise, noise_sr = torchaudio.load(noise_path)

            if noise_sr != 16000:
                noise = torchaudio.functional.resample(
                    noise, noise_sr, 16000
                )

            snr = rng.choice(self.snrs)

            speech = mix_with_noise(
                speech,
                noise,
                snr_db=snr
            )

        return {
            "speech": speech.squeeze(),
            "text": sample["text"]
        }

# Collate Functions

In [8]:
def train_collate_fn(batch):
    speeches = [sample["speech"].numpy() for sample in batch]

    texts = [sample["text"] for sample in batch]

    # convert audio
    inputs = processor(speeches, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors="pt")

    # tokenize transcript
    labels = processor.tokenizer(texts, padding=True, return_tensors="pt")

    label_ids = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)

    return {"input_values": inputs.input_values, "attention_mask": inputs.attention_mask, "labels": label_ids}

In [9]:
def eval_collate_fn(batch):
    speeches = [sample["speech"].numpy() for sample in batch]

    inputs = processor(speeches, sampling_rate=16000, padding=True, return_attention_mask = True, return_tensors="pt")

    texts = [sample["text"] for sample in batch]

    return inputs, texts

# Word Error Rate Baseline

In [10]:
%%time
def calc_error_rate(model, dataloader):
    predictions = []
    references = []

    model.eval()

    with torch.no_grad():
        for inputs, texts in dataloader:
            logits = model(input_values=inputs.input_values.to(device), attention_mask=inputs.attention_mask.to(device)).logits

            predicted_ids = torch.argmax(logits, dim=-1)

            transcriptions = processor.batch_decode(predicted_ids.cpu())

            predictions.extend(transcriptions)
            references.extend(texts)

    return wer(references, predictions)
base_test_dataset = SpeechDataset(samples = test_samples)
test_loader = DataLoader(base_test_dataset, batch_size = 8, shuffle=False, collate_fn = eval_collate_fn)
# print(f"Baseline WER no noise: {calc_error_rate(model, test_loader)}")

CPU times: total: 0 ns
Wall time: 0 ns


Baseline WER no noise: 0.17015368228849664

In [11]:
SNRs = [-5, 0, 5, 10]
test_snr_loaders = {}
for s in SNRs:
    dataset = SpeechDataset(samples = test_samples, noise_files = noise_files, snrs = [s], augment = True)
    loader = DataLoader(dataset, batch_size = 8, shuffle=False, collate_fn = eval_collate_fn)
    test_snr_loaders[s] = loader

# for s in test_snr_loaders:
#     error_rate = calc_error_rate(model, test_snr_loaders[s])
#     print(f"Baseline WER at SNR {s}: {error_rate:.4f}")

Baseline WER at SNR -5: 0.8499

Baseline WER at SNR 0: 0.6684

Baseline WER at SNR 5: 0.4390

Baseline WER at SNR 10: 0.2896

# LLM Rescoring

In [12]:
#build ctc decoder
from pyctcdecode import build_ctcdecoder

vocab_dict = processor.tokenizer.get_vocab()

labels = [token for token, idx in sorted(vocab_dict.items(), key = lambda x: x[1])]

decoder = build_ctcdecoder(labels)

kenlm python bindings are not installed. Most likely you want to install it using: pip install https://github.com/kpu/kenlm/archive/master.zip
kenlm python bindings are not installed. Most likely you want to install it using: pip install https://github.com/kpu/kenlm/archive/master.zip
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


In [13]:
#score for language model
def get_lm_score(text):
    inputs = lm_tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = lm_model(**inputs, labels=inputs["input_ids"])

    num_tokens = inputs["input_ids"].shape[1] - 1

    log_prob = -outputs.loss.item() * num_tokens

    return log_prob

In [14]:
def calc_error_rate_llm(model, dataloader, alpha=.1):
    predictions = []
    references = []

    model.eval()

    with torch.no_grad():
        for inputs, texts in dataloader:
            logits = model(input_values=inputs.input_values.to(device), attention_mask=inputs.attention_mask.to(device)).logits

            for i in range(logits.shape[0]):

                logits_np = logits[i].cpu().numpy()

                beams = decoder.decode_beams(logits_np, beam_width=10)

                #top 5 answers from ctc search
                best_beams = [beam for beam in beams if beam[0].strip()][:5]

                best_text = ""
                best_score = float("-inf")

                for beam in best_beams:

                    # Check your pyctcdecode beam structure!
                    text = beam[0]
                    ctc_score = beam[3]

                    lm_score = get_lm_score(text)

                    combined_score = (ctc_score + alpha * lm_score)

                    if combined_score > best_score:
                        best_score = combined_score
                        best_text = text

                predictions.append(best_text)

            references.extend(texts)

    return wer(references, predictions)

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

lm_name = "openai-community/gpt2"

lm_tokenizer = AutoTokenizer.from_pretrained(lm_name)
lm_model = AutoModelForCausalLM.from_pretrained(lm_name).to(device)
_ = lm_model.eval()

In [16]:
#tune alpha for LM rescoring (use validation datasets for fine tuning, not enough space on device for train val and test dataset)
# alpha_samples = dev_samples[:200]
# dummy_dataset = SpeechDataset(samples = alpha_samples)
# dummy_loader = DataLoader(dummy_dataset, batch_size = 8, shuffle=False, collate_fn = eval_collate_fn)
# alphas = [.0, .1, .2, .3]
# for a in alphas:
#     print(f"LLM Rescoring WER no noise at alpha {a}: {calc_error_rate_llm(model, dummy_loader, a)}")

LLM Rescoring WER no noise at alpha 0.0: 0.16185765320142897

LLM Rescoring WER no noise at alpha 0.1: 0.15883484473756526

LLM Rescoring WER no noise at alpha 0.2: 0.15828524319868095

LLM Rescoring WER no noise at alpha 0.3: 0.15856004396812312

In [17]:
# print(f"LLM Rescoring WER no noise: {calc_error_rate_llm(model, test_loader)}")

LLM Rescoring WER no noise alpha = 1: 0.16754793061472914

In [18]:
# %%time
# for s in test_snr_loaders:
#     error_rate = calc_error_rate_llm(model, test_snr_loaders[s])
#     print(f"LLM Rescoring WER at SNR {s}: {error_rate:.4f}")

alpha = .1

LLM Rescoring WER at SNR -5: 0.8771

LLM Rescoring WER at SNR 0: 0.6929

LLM Rescoring WER at SNR 5: 0.4738

LLM Rescoring WER at SNR 10: 0.3061

# Fine-Tuning With Noisy Data

In [19]:
# train_dataset = SpeechDataset(samples=dev_samples, noise_files=noise_files, snrs=[-5, 0, 5, 10], augment=True)

# train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=train_collate_fn, num_workers=0, pin_memory = True)

# optimizer = AdamW(model.parameters(), lr=1e-5)

# model.freeze_feature_encoder()

In [20]:
# num_epochs = 1

# model.train()

# for epoch in range(num_epochs):

#     total_loss = 0

#     for batch in train_loader:

#         input_values = batch["input_values"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"].to(device)

#         optimizer.zero_grad()

#         outputs = model(input_values=input_values, attention_mask=attention_mask, labels=labels)

#         loss = outputs.loss

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     avg_loss = total_loss / len(train_loader)


In [21]:
#save
# save_path = "./fine_tuned_model"

# model.save_pretrained(save_path)
# processor.save_pretrained(save_path)

In [22]:
# #load noise tuned model
# model_path = r"./fine_tuned_model_Noise"

# noisy_processor = Wav2Vec2Processor.from_pretrained(model_path)
# noisy_model = Wav2Vec2ForCTC.from_pretrained(model_path)

# noisy_model = model.to(device)
# _ = noisy_model.eval()

In [23]:
# print(f"Noisy Tuning WER no noise: {calc_error_rate(model, test_loader)}")
# for s in test_snr_loaders:
#     error_rate = calc_error_rate(model, test_snr_loaders[s])
#     print(f"Noisy Tuning WER at SNR {s}: {error_rate:.4f}")

Noisy Tuning WER no noise: 0.051943852708460136

Noisy Tuning WER at SNR -5: 0.6841

Noisy Tuning WER at SNR 0: 0.4109

Noisy Tuning WER at SNR 5: 0.1954

Noisy Tuning WER at SNR 10: 0.1015

In [24]:
# print(f"LLM Rescoring Noisy Fine tuning WER no noise: {calc_error_rate_llm(model, test_loader)}")


LLM Rescoring Noisy Fine tuning WER no noise: 0.04922398052343274

In [25]:
# for s in test_snr_loaders:
#     error_rate = calc_error_rate_llm(model, test_snr_loaders[s])
#     print(f"LLM Rescoring Noisy Fine tuning WER at SNR {s}: {error_rate:.4f}")

LLM Rescoring Noisy Fine tuning WER at SNR -5: 0.6945

LLM Rescoring Noisy Fine tuning WER at SNR 0: 0.4281

LLM Rescoring Noisy Fine tuning WER at SNR 5: 0.2067

LLM Rescoring Noisy Fine tuning WER at SNR 10: 0.0991

# Adversarial Training

In [26]:
def fgsm_attack(model, input_values, attention_mask, labels, epsilon=0.001):
    
    adv_input = input_values.clone().detach().requires_grad_(True)

    # forward
    outputs = model(
        input_values=adv_input,
        attention_mask=attention_mask,
        labels=labels
    )
    loss = outputs.loss

    # fgsm
    data_grad = torch.autograd.grad(loss, adv_input, retain_graph=False, create_graph = False)[0]
    perturbed = adv_input.detach() + epsilon * data_grad.sign().detach()

    #clip
    perturbed = torch.clamp(perturbed, -1.0, 1.0)

    return perturbed.detach()

In [27]:
# train_dataset = SpeechDataset(samples=dev_samples, noise_files=noise_files, snrs=[-5, 0, 5, 10], augment=False)

# train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=train_collate_fn, num_workers=0, pin_memory = True)

# #load
# processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
# model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

# model = model.to(device)

# optimizer = AdamW(model.parameters(), lr=1e-5)

# model.freeze_feature_encoder()

# num_epochs = 1

# model.train()

# for epoch in range(num_epochs):

#     total_loss = 0

#     for batch in train_loader:

#         input_values = batch["input_values"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"].to(device)

#         attack = fgsm_attack(model, input_values, attention_mask, labels)

#         optimizer.zero_grad()

#         outputs = model(input_values=attack, attention_mask=attention_mask, labels=labels)

#         loss = outputs.loss

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     avg_loss = total_loss / len(train_loader)


In [28]:
#save
# save_path = "./adversarial_fine_tuned_model"

# model.save_pretrained(save_path)
# processor.save_pretrained(save_path)

In [29]:
# print(f"Adversarial Tuning WER no noise: {calc_error_rate(model, test_loader)}")
# for s in test_snr_loaders:
#     error_rate = calc_error_rate(model, test_snr_loaders[s])
#     print(f"Adversarial Tuning WER at SNR {s}: {error_rate:.4f}")

Adversarial Tuning WER no noise: 0.046466068167985396

Adversarial Tuning WER at SNR -5: 0.7704

Adversarial Tuning WER at SNR 0: 0.5105

Adversarial Tuning WER at SNR 5: 0.2343

Adversarial Tuning WER at SNR 10: 0.1097

In [30]:
# print(f"LLM Rescoring Noisy Fine tuning WER no noise: {calc_error_rate_llm(model, test_loader)}")
# for s in test_snr_loaders:
#     error_rate = calc_error_rate_llm(model, test_snr_loaders[s])
#     print(f"LLM Rescoring Noisy Fine tuning WER at SNR {s}: {error_rate:.4f}")

LLM Rescoring Noisy Fine tuning WER no noise: 0.04262401095556908

LLM Rescoring Noisy Fine tuning WER at SNR -5: 0.7824

LLM Rescoring Noisy Fine tuning WER at SNR 0: 0.5059

LLM Rescoring Noisy Fine tuning WER at SNR 5: 0.2413

LLM Rescoring Noisy Fine tuning WER at SNR 10: 0.1086

# Noisy + Adversarial

In [31]:
# train_dataset = SpeechDataset(samples=dev_samples, noise_files=noise_files, snrs=[-5, 0, 5, 10], augment=True)

# train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=train_collate_fn, num_workers=0, pin_memory = True)

# #load
# processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
# model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

# model = model.to(device)

# optimizer = AdamW(model.parameters(), lr=1e-5)

# model.freeze_feature_encoder()

# num_epochs = 1

# model.train()

# for epoch in range(num_epochs):

#     total_loss = 0

#     for batch in train_loader:

#         input_values = batch["input_values"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"].to(device)

#         attack = fgsm_attack(model, input_values, attention_mask, labels)

#         optimizer.zero_grad()

#         outputs = model(input_values=attack, attention_mask=attention_mask, labels=labels)

#         loss = outputs.loss

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     avg_loss = total_loss / len(train_loader)


In [32]:
# #save
# save_path = "./noisy_adversarial_fine_tuned_model"

# model.save_pretrained(save_path)
# processor.save_pretrained(save_path)

In [33]:
# print(f"Noisy Adversarial Tuning WER no noise: {calc_error_rate(model, test_loader)}")
# for s in test_snr_loaders:
#     error_rate = calc_error_rate(model, test_snr_loaders[s])
#     print(f"Noisy Adversarial Tuning WER at SNR {s}: {error_rate:.4f}")

Noisy Adversarial Tuning WER no noise: 0.05673691418137553

Noisy Adversarial Tuning WER at SNR -5: 0.6958

Noisy Adversarial Tuning WER at SNR 0: 0.4216

Noisy Adversarial Tuning WER at SNR 5: 0.1993

Noisy Adversarial Tuning WER at SNR 10: 0.1059

In [34]:
# print(f"LLM Rescoring Noisy adversarial Fine tuning WER no noise: {calc_error_rate_llm(model, test_loader)}")
# for s in test_snr_loaders:
#     error_rate = calc_error_rate_llm(model, test_snr_loaders[s])
#     print(f"LLM Rescoring Noisy adversarial Fine tuning WER at SNR {s}: {error_rate:.4f}")

LLM Rescoring Noisy adversarial Fine tuning WER no noise: 0.054264303104077906

LLM Rescoring Noisy adversarial Fine tuning WER at SNR -5: 0.7007

LLM Rescoring Noisy adversarial Fine tuning WER at SNR 0: 0.4354

LLM Rescoring Noisy adversarial Fine tuning WER at SNR 5: 0.2074

LLM Rescoring Noisy adversarial Fine tuning WER at SNR 10: 0.1063